# 03_train_hyperspeech
Trains HyperSpeech TokenMixer by fold and stores calibrated predictions and metrics.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.data import load_dataset
from src.pipeline import run_hyperspeech_tokenmixer_fold
from src.splits import load_splits

bundle = load_dataset(
    csv_path=ROOT / "data" / "all_data_clean.csv",
    target_col="sbp_binary",
    subject_col="PAT_ID",
)
splits = load_splits(ROOT / "artifacts" / "splits" / "outer_splits.json")

results = []
for fold_info in splits:
    fold = fold_info["fold"]
    result = run_hyperspeech_tokenmixer_fold(
        df=bundle.df,
        feature_cols=bundle.feature_cols,
        target_col=bundle.target_col,
        subject_col=bundle.subject_col,
        train_idx=fold_info["train_idx"],
        test_idx=fold_info["test_idx"],
        fold=fold,
        out_dir=ROOT / "artifacts" / "metrics",
        device="cpu",
    )
    results.append(result)
    print(f"hyperspeech_tokenmixer fold {fold}: ok")

pd.DataFrame(results).head()